In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader


In [2]:
# #photos seperation 
# import os
# import shutil

# source_dir = './train/'
# cats_dir = './cats/'
# dogs_dir = './dogs/'

# for filename in os.listdir(source_dir):
#     source_path = os.path.join(source_dir, filename)

#     if os.path.isfile(source_path):
#         if "cat" in filename.lower():
#             destination_dir = cats_dir
#         elif "dog" in filename.lower():
#             destination_dir = dogs_dir
#         else:
#             print(f"skipping {filename} as it does not contain 'cat' or 'dog'")
#             continue
#         destination_path = os.path.join(destination_dir, filename)

#         try :
#             shutil.move(source_path, destination_path)
#             print(f"Moved {filename} to {destination_dir}")
#         except shutil.error as e:
#             print(f"Error moving {filename}: {e}")
# print("Image separation complete.")

In [4]:
#DATASETS AND DATALOADER
print(torch.__version__) 
train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# Combined dataset for cats and dogs
train_dataset = datasets.ImageFolder(root="./data/", transform=train_transforms)

BatchSize = 32
train_loader = DataLoader(train_dataset, batch_size=BatchSize, shuffle=True)

#Building the CNN model

class CatDogCNN(nn.Module):
    def __init__(self):
        super(CatDogCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 16 * 16, 512)
        self.fc2 = nn.Linear(512, 2) #two classes: cat and dog
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        # Flatten the output for the fully connected layers
        x = x.view(-1, 128 * 16 * 16)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

#initiate the model 
model = CatDogCNN()

2.11.0


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

#loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        #forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        #backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training Complete.")





Epoch [1/10], Loss: 0.0004
Epoch [1/10], Loss: 0.0005
Epoch [1/10], Loss: 0.0006
Epoch [1/10], Loss: 0.0006
Epoch [1/10], Loss: 0.0007
Epoch [1/10], Loss: 0.0008
Epoch [1/10], Loss: 0.0009
Epoch [1/10], Loss: 0.0009
Epoch [1/10], Loss: 0.0009
Epoch [1/10], Loss: 0.0010
Epoch [1/10], Loss: 0.0010
Epoch [1/10], Loss: 0.0010
Epoch [1/10], Loss: 0.0011
Epoch [1/10], Loss: 0.0012
Epoch [1/10], Loss: 0.0013
Epoch [1/10], Loss: 0.0013
Epoch [1/10], Loss: 0.0014
Epoch [1/10], Loss: 0.0016
Epoch [1/10], Loss: 0.0017
Epoch [1/10], Loss: 0.0018
Epoch [1/10], Loss: 0.0018
Epoch [1/10], Loss: 0.0018
Epoch [1/10], Loss: 0.0018
Epoch [1/10], Loss: 0.0019
Epoch [1/10], Loss: 0.0019
Epoch [1/10], Loss: 0.0019
Epoch [1/10], Loss: 0.0019
Epoch [1/10], Loss: 0.0020
Epoch [1/10], Loss: 0.0020
Epoch [1/10], Loss: 0.0021
Epoch [1/10], Loss: 0.0021
Epoch [1/10], Loss: 0.0022
Epoch [1/10], Loss: 0.0022
Epoch [1/10], Loss: 0.0022
Epoch [1/10], Loss: 0.0022
Epoch [1/10], Loss: 0.0023
Epoch [1/10], Loss: 0.0023
E

In [12]:
torch.save(model.state_dict(), "cat_dog_model.pth")

In [5]:
import os 
from PIL import Image
import pandas as pd
import numpy as np


In [6]:
from torch.utils.data import Dataset
class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.images = [f for f in os.listdir(test_dir) if f.endswith('.jpg')]
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.test_dir, img_name)

        #load the image
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        img_id = int(img_name.split('.')[0])
        return image, img_id



In [7]:
model.load_state_dict(torch.load("cat_dog_model.pth"))
model.eval() #most impt

test_dir = "./test/"
test_dataset = TestDataset(test_dir = test_dir, transform = train_transforms)
test_loader = DataLoader(test_dataset, batch_size=BatchSize, shuffle=False)

all_ids = []
all_preds = []

print("Starting Inference on Test Set...")
with torch.no_grad():
    for images, img_ids in test_loader:
        outputs = model(images)
        probabilities = F.softmax(outputs, dim=1)
        predicted = probabilities[:, 1]  # probability of being a dog (class 1)

        all_ids.extend(img_ids.numpy())
        all_preds.extend(predicted.detach().numpy())

print("prediction is completed.")

Starting Inference on Test Set...
prediction is completed.


In [8]:
submission = pd.DataFrame({"id": all_ids, "label": all_preds})
submission = submission.sort_values(by = 'id')

submission.to_csv('cnn_submission.csv', index=False)
print("Saved to cnn_submission.csv")

Saved to cnn_submission.csv


Using ResNet18 for better prediction, often used in ml industry to not train model from scratch but use pretrained model adn transfer it to your dataset
Now few things to follow like steps 
freezing pre-trained weights, 
replacing the final fully connected layer to output two classes (cat/dog), 
and fine-tuning on a labeled dataset.
ResNet takes 224, 224 image dim so transforming training and test dataset before passing it to the model training and testing 

In [26]:
#transforms.resize(224, 224) for resnet
train_transforms_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Combined dataset for cats and dogs
train_dataset_2 = datasets.ImageFolder(root="./data/", transform=train_transforms_resnet)

BatchSize = 8  # Further reduced for faster CPU training
train_loader_2 = DataLoader(train_dataset_2, batch_size=BatchSize, shuffle=True)


In [27]:
#import resnet model
from torchvision import models
resnet_model = models.resnet18(weights=None)  # Don't download automatically
resnet_model.load_state_dict(torch.load('resnet18-5c106cde.pth', weights_only=False))  # Load from downloaded file (trusted source)
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, 2) #two classes

print("Before freezing:")
trainable_before = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f"Trainable parameters before freeze: {trainable_before}")

# Freeze all layers except the final FC layer
for param in resnet_model.parameters():
    param.requires_grad = False

print("After freezing all:")
trainable_after_freeze = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f"Trainable parameters after freeze: {trainable_after_freeze}")

# Unfreeze the FC layer parameters
for param in resnet_model.fc.parameters():
    param.requires_grad = True

print("After unfreezing FC:")
trainable_after_unfreeze = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f"Trainable parameters after unfreeze: {trainable_after_unfreeze}")

# Verify freezing
trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in resnet_model.parameters())
print(f"Final trainable parameters: {trainable_params}")
print(f"Total parameters: {total_params}")
print(f"Only training {100 * trainable_params / total_params:.2f}% of parameters")


Before freezing:
Trainable parameters before freeze: 11177538
After freezing all:
Trainable parameters after freeze: 0
After unfreezing FC:
Trainable parameters after unfreeze: 1026
Final trainable parameters: 1026
Total parameters: 11177538
Only training 0.01% of parameters


In [28]:
#training the new model for final layer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet_model = resnet_model.to(device)  # Reassign to ensure it's on device

#loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=0.001)  # Only optimize trainable params

#training loop
num_epochs = 5
for epoch in range(num_epochs):
    resnet_model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader_2):
        inputs, labels = inputs.to(device), labels.to(device)

        #forward pass
        outputs = resnet_model(inputs)
        loss = criterion(outputs, labels)

        #backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader_2):.4f}")

print("Training Complete.")


Epoch [1/5], Loss: 0.0003
Epoch [1/5], Loss: 0.0006
Epoch [1/5], Loss: 0.0008
Epoch [1/5], Loss: 0.0010
Epoch [1/5], Loss: 0.0013
Epoch [1/5], Loss: 0.0015
Epoch [1/5], Loss: 0.0016
Epoch [1/5], Loss: 0.0018
Epoch [1/5], Loss: 0.0020
Epoch [1/5], Loss: 0.0022
Epoch [1/5], Loss: 0.0023
Epoch [1/5], Loss: 0.0025
Epoch [1/5], Loss: 0.0026
Epoch [1/5], Loss: 0.0028
Epoch [1/5], Loss: 0.0029
Epoch [1/5], Loss: 0.0030
Epoch [1/5], Loss: 0.0031
Epoch [1/5], Loss: 0.0033
Epoch [1/5], Loss: 0.0034
Epoch [1/5], Loss: 0.0036
Epoch [1/5], Loss: 0.0038
Epoch [1/5], Loss: 0.0039
Epoch [1/5], Loss: 0.0041
Epoch [1/5], Loss: 0.0042
Epoch [1/5], Loss: 0.0043
Epoch [1/5], Loss: 0.0044
Epoch [1/5], Loss: 0.0046
Epoch [1/5], Loss: 0.0047
Epoch [1/5], Loss: 0.0048
Epoch [1/5], Loss: 0.0050
Epoch [1/5], Loss: 0.0051
Epoch [1/5], Loss: 0.0052
Epoch [1/5], Loss: 0.0053
Epoch [1/5], Loss: 0.0055
Epoch [1/5], Loss: 0.0056
Epoch [1/5], Loss: 0.0057
Epoch [1/5], Loss: 0.0058
Epoch [1/5], Loss: 0.0061
Epoch [1/5],

In [29]:
# Save the trained ResNet model
torch.save(resnet_model.state_dict(), "resnet_cat_dog_model.pth")
print("ResNet model saved to resnet_cat_dog_model.pth")

# Inference with ResNet
resnet_model.load_state_dict(torch.load("resnet_cat_dog_model.pth"))
resnet_model.eval()

test_dir = "./test/"
test_dataset_resnet = TestDataset(test_dir=test_dir, transform=train_transforms_resnet)
test_loader_resnet = DataLoader(test_dataset_resnet, batch_size=BatchSize, shuffle=False)

all_ids = []
all_preds = []

print("Starting Inference on Test Set with ResNet...")
with torch.no_grad():
    for images, img_ids in test_loader_resnet:
        images = images.to(device)
        outputs = resnet_model(images)
        probabilities = F.softmax(outputs, dim=1)
        predicted = probabilities[:, 1]  # probability of being a dog (class 1)

        all_ids.extend(img_ids.numpy())
        all_preds.extend(predicted.detach().cpu().numpy())

print("ResNet prediction completed.")

# Create submission
submission_resnet = pd.DataFrame({"id": all_ids, "label": all_preds})
submission_resnet = submission_resnet.sort_values(by='id')
submission_resnet.to_csv('resnet_submission.csv', index=False)
print("Saved ResNet predictions to resnet_submission.csv")


ResNet model saved to resnet_cat_dog_model.pth
Starting Inference on Test Set with ResNet...
ResNet prediction completed.
Saved ResNet predictions to resnet_submission.csv
